# 5. Học không giám sát – Phân cụm bài báo theo chủ đề

**Mục tiêu:** nhóm bài theo nội dung, giảm các cụm chỉ phản ánh video/văn nói và cung cấp bằng chứng kiểm tra chất lượng cụm.

Sử dụng **TF-IDF hướng chủ đề (unigram + bigram, lọc từ văn phong) → LSA 100 chiều → K-Means**. Khảo sát nhiều K, xem các chỉ số và chọn K thủ công.

Nhãn `label` và `subject` chỉ dùng sau khi chốt mô hình. Các hàm nằm trong `topic_clustering.py`; bộ lọc ở đây chỉ dành cho phân cụm, không sửa dữ liệu hay mô hình phân loại.


In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from threadpoolctl import threadpool_limits
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_samples
from scipy.stats import chi2_contingency

from topic_clustering import (
    TOPIC_STOP_WORDS, build_features, evaluate_labels, search_clusters,
    keyword_table, article_tables, fitted_pipeline, predict_topics,
)

SEED = 42
SEEDS = (42, 17, 93)
K_VALUES = (6, 8, 10, 12, 14)
N_INIT = 10
SAMPLE_SIZE = 3000
DIMENSIONS = 100
RESULTS_DIR = Path("results/clustering")
MODELS_DIR = Path("models")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv("data/processed/news_clean.csv", keep_default_na=False)
assert df["raw_id"].is_unique, "raw_id phải duy nhất để đối chiếu bài báo."
print(f"Đọc {len(df):,} bài. Chọn mô hình chỉ từ clean_text.")

## 1. Biểu diễn văn bản và mẫu đánh giá

- **TF-IDF hướng chủ đề:** tối đa 30.000 đặc trưng unigram + bigram, `min_df=10`, `max_df=0.4`; lọc stop-word tiếng Anh cùng danh sách cố định từ văn nói/định dạng như `say`, `get`, `video`, `watch`, `via`.
- Bigram như `north korea`, `supreme court` giữ ý nghĩa cụm từ. Bộ lọc là giả định thiết kế: `video`, `twitter`, `president` cũng có thể mang nội dung; danh sách được xuất để kiểm tra và thử nghiệm thêm.
- Bài không còn đặc trưng được ghi riêng, gán `cluster=-1`, không ép vào một chủ đề.
- Giảm chiều bằng LSA xuống 100 chiều, chuẩn hóa vector và dùng float64.

Các K được đánh giá trên **cùng mẫu ngẫu nhiên cố định và cùng không gian TF-IDF trước LSA** để tính cosine silhouette. Silhouette Euclidean sau LSA được báo riêng; không so hai loại điểm như cùng một thước đo.


In [ ]:
with threadpool_limits(limits=4):
    topic_features = build_features(df["clean_text"], topic_focused=True,
                                    seed=SEED, dimensions=DIMENSIONS)

valid = topic_features["valid"]
work = df.loc[valid].reset_index(drop=True)
excluded = df.loc[~valid, ["raw_id", "title"]].copy()
excluded["reason"] = "No TF-IDF features after filtering"
excluded.to_csv(RESULTS_DIR / "excluded_articles.csv", index=False)

X = topic_features["dense"][valid]
X_tfidf = topic_features["sparse"][valid]
sample_indices = np.sort(np.random.default_rng(SEED).choice(
    len(work), size=min(SAMPLE_SIZE, len(work)), replace=False))
reference_sample = X_tfidf[sample_indices]
work.iloc[sample_indices][["raw_id"]].to_csv(RESULTS_DIR / "evaluation_sample.csv", index=False)
(RESULTS_DIR / "topic_stop_words.txt").write_text("\n".join(TOPIC_STOP_WORDS), encoding="utf-8")

print(f"Phân cụm {len(work):,} bài; loại {len(excluded)} bài không có đặc trưng.")
print(f"Mẫu đánh giá: {len(sample_indices):,} bài; LSA: {DIMENSIONS} chiều.")
print("TF-IDF hướng chủ đề:", topic_features["sparse"].shape)
print("Tỷ lệ phương sai LSA giữ lại:",
      round(float(topic_features["svd"].explained_variance_ratio_.sum()), 4))


## 2. Khảo sát K và chọn thủ công

Mỗi K trong `K_VALUES` chạy với các seed trong `SEEDS`, mỗi seed dùng `N_INIT` lần khởi tạo.

Chạy cell khảo sát để xem bảng tổng hợp, kết quả từng seed và ba biểu đồ, sau đó tự đặt `K` trong cell chọn bên dưới. Cân nhắc:
- **Cosine silhouette TF-IDF:** cao hơn thể hiện các cụm tách biệt hơn; xem thêm độ lệch chuẩn giữa seed.
- **Stability ARI:** cao hơn thể hiện kết quả ít thay đổi theo seed. Đây là ARI giữa các lần phân cụm, không dùng Fake/True.
- **Min/Max cluster share:** kiểm tra cụm quá nhỏ hoặc một cụm chiếm phần lớn dữ liệu.
- **Inertia:** tham khảo điểm khuỷu tay; inertia thường giảm khi tăng K.

Không áp dụng ngưỡng loại hay quy tắc chọn K tự động. Bạn tự giải thích sự đánh đổi giữa các chỉ số khi thuyết trình. Giữ đúng mô hình của seed đầu tiên đã đánh giá, không fit lại.


In [ ]:
with threadpool_limits(limits=4):
    k_scores, k_runs, candidate_models = search_clusters(
        X, reference_sample, sample_indices, k_values=K_VALUES,
        seeds=SEEDS, n_init=N_INIT)

# Chưa chốt K: xuất và hiển thị các chỉ số để tự so sánh.
k_scores.to_csv(RESULTS_DIR / "k_selection.csv")
k_runs.to_csv(RESULTS_DIR / "k_runs.csv", index=False)
print("Bảng tổng hợp theo K:")
display(k_scores.round(5))
print("Kết quả chi tiết từng seed:")
display(k_runs.round(5))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].errorbar(k_scores.index, k_scores["Silhouette TF-IDF cosine"],
                 yerr=k_scores["Silhouette TF-IDF std"], marker="o", capsize=4)
axes[0].set(title="Cosine silhouette trước LSA", xlabel="K", ylabel="Silhouette (mẫu cố định)")
axes[1].plot(k_scores.index, k_scores["Stability ARI"], marker="o")
axes[1].set(title=f"Độ ổn định qua {len(SEEDS)} seed", xlabel="K", ylabel="ARI giữa các lần chạy", ylim=(0, 1.05))
axes[2].plot(k_scores.index, k_scores["Inertia"], marker="o")
axes[2].set(title="Inertia trung bình (tham khảo)", xlabel="K", ylabel="Inertia")
plt.tight_layout()
fig.savefig(RESULTS_DIR / "k_selection.png", dpi=150, bbox_inches="tight")
plt.show()

### Chọn K thủ công sau khi xem kết quả

Sửa `K = 10` trong cell dưới thành giá trị bạn chọn từ bảng khảo sát. Giá trị 10 được giữ để bạn chỉnh sửa, không phải kết luận tự động.

**Lý do chọn K (bạn bổ sung để thuyết trình):** nêu các giá trị silhouette, độ ổn định, tỷ lệ cụm và mức giảm inertia; giải thích vì sao lựa chọn này phù hợp hơn các K lân cận.

Sau khi đổi K, chạy lại cell chọn và toàn bộ các cell phía sau để cập nhật nhãn cụm, bảng, hình và mô hình lưu.


In [ ]:
K = 10  # Tự sửa sau khi xem bảng chỉ số và biểu đồ ở trên.
if K not in candidate_models:
    raise ValueError(f"K={K} chưa được khảo sát. Chọn trong {list(candidate_models)} hoặc sửa K_VALUES rồi chạy lại.")

kmeans = candidate_models[K]
work["cluster"] = kmeans.labels_
k_scores["Selected"] = k_scores.index == K
k_scores.to_csv(RESULTS_DIR / "k_selection.csv")
print(f"Chọn thủ công K={K}; dùng candidate seed={SEEDS[0]}, n_init={N_INIT}.")
display(k_scores.round(5))


## 3. Đánh giá mô hình đã chọn

Bảng dưới ghi các chỉ số của đúng mô hình đã chọn thủ công, lưu vào `final_metrics.csv`. Cosine silhouette dùng mẫu TF-IDF cố định trước LSA; silhouette Euclidean được tính trong LSA. ARI ổn định là giá trị tổng hợp giữa các seed tại K đã chọn.

`k_selection.csv` ghi các chỉ số tổng hợp cho từng K; `k_runs.csv` ghi kết quả từng seed.


In [ ]:
with threadpool_limits(limits=4):
    selected_metrics = evaluate_labels(X, reference_sample, kmeans.labels_, sample_indices)
final_metrics = pd.DataFrame([{
    "Model": f"K-Means, K={K}",
    "K": K,
    **selected_metrics,
    "Stability ARI": k_scores.loc[K, "Stability ARI"],
    "Inertia": kmeans.inertia_,
}]).set_index("Model")
final_metrics.to_csv(RESULTS_DIR / "final_metrics.csv")
display(final_metrics.round(5))


## 4. Từ khóa có trọng số và bài đại diện

Xếp từ khóa theo **TF-IDF trung bình × log(1 + lift)**; lift là tỷ lệ TF-IDF trung bình trong cụm so với toàn bộ dữ liệu. Chỉ giữ từ xuất hiện trong ≥ 1% bài của cụm để giảm từ quá hiếm. CSV lưu cả trọng số và tỷ lệ bài chứa từ.

Mỗi cụm có ba bài gần tâm nhất để kiểm tra nội dung. Tên gợi ý từ ba từ khóa chỉ hỗ trợ đọc; cần đối chiếu bài đại diện trước khi đặt tên chủ đề chính thức.

In [ ]:
terms = topic_features["tfidf"].get_feature_names_out()
keywords = keyword_table(X_tfidf, terms, work["cluster"].to_numpy())
articles, representatives, boundary = article_tables(work, X, kmeans)
summary = work.groupby("cluster").size().to_frame("Số bài")
summary["% dữ liệu"] = summary["Số bài"] / len(work) * 100
summary["Chủ đề gợi ý"] = keywords[keywords["rank"] <= 3].groupby("cluster")["term"].agg(" / ".join)
summary["Từ khóa"] = keywords[keywords["rank"] <= 10].groupby("cluster")["term"].agg(", ".join)
summary["Bài gần tâm nhất"] = representatives[representatives["rank"] == 1].set_index("cluster")["title"]
keywords.to_csv(RESULTS_DIR / "cluster_keywords.csv", index=False)
representatives.to_csv(RESULTS_DIR / "representative_articles.csv", index=False)
boundary.to_csv(RESULTS_DIR / "boundary_articles.csv", index=False)

assignments = df[["raw_id", "title"]].merge(
    articles.drop(columns="title"), on="raw_id", how="left", validate="one_to_one")
assignments["cluster"] = assignments["cluster"].fillna(-1).astype(int)
assignments.to_csv(RESULTS_DIR / "cluster_articles.csv", index=False)
summary.to_csv(RESULTS_DIR / "cluster_summary.csv")
with pd.option_context("display.max_colwidth", 110):
    display(summary.round(2))
    display(representatives[["cluster", "rank", "raw_id", "title", "distance_to_center"]])

## 5. Bài ở ranh giới và hình chiếu minh họa

- `boundary_margin` là khoảng cách tới tâm gần thứ hai trừ khoảng cách tới tâm được gán. Margin nhỏ nghĩa là gần ranh giới; **không phải xác suất hay độ tin cậy đã hiệu chuẩn**.
- PCA 2D chỉ minh họa trên mẫu đánh giá. Không phân cụm trên hình 2D hoặc dùng độ tách trên hình làm bằng chứng chất lượng.
- Phân bố silhouette theo cụm giúp phát hiện cụm còn lẫn; giá trị âm nghĩa là bài gần cụm khác hơn theo khoảng cách trung bình.

In [ ]:
with threadpool_limits(limits=4):
    projection = PCA(n_components=2, random_state=SEED).fit_transform(X[sample_indices])
    sample_silhouette = silhouette_samples(reference_sample, kmeans.labels_[sample_indices], metric="cosine")
sample_labels = kmeans.labels_[sample_indices]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
summary["Số bài"].plot.bar(ax=axes[0], color="#2a78d6", rot=0)
axes[0].set(title="Kích thước cụm", xlabel="Cụm", ylabel="Số bài")
scatter = axes[1].scatter(projection[:, 0], projection[:, 1], c=sample_labels,
                          cmap="tab20", s=5, alpha=.35, vmin=-.5, vmax=K-.5)
axes[1].set(title="PCA 2D – chỉ minh họa", xlabel="PC1", ylabel="PC2")
fig.colorbar(scatter, ax=axes[1], ticks=range(K), label="Cụm")
axes[2].boxplot([sample_silhouette[sample_labels == c] for c in range(K)],
                tick_labels=list(range(K)), showfliers=False)
axes[2].axhline(0, color="grey", ls=":")
axes[2].set(title="Cosine silhouette theo cụm", xlabel="Cụm", ylabel="Silhouette trước LSA")
plt.tight_layout()
fig.savefig(RESULTS_DIR / "cluster_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("Các bài có margin thấp, cần đọc để kiểm tra ranh giới:")
with pd.option_context("display.max_colwidth", 110):
    display(boundary[["cluster", "raw_id", "title", "boundary_margin"]])

## 6. Đối chiếu với nhãn sau khi chốt mô hình

Đến đây mới dùng Fake/True và chuyên mục gốc. ARI với nhãn, Cramér's V và purity mô tả liên hệ giữa cụm và nhãn; không phải tiêu chí chọn K và không xác nhận chủ đề đúng. Chuyên mục gốc cũng gắn với nguồn đăng nên không xem là nhãn chủ đề chuẩn.

ARI thấp không chứng minh độc lập với Fake/True. Số cụm và số lớp khác nhau ảnh hưởng việc so hai cách chia nhóm; purity có xu hướng tăng khi tăng K.

In [ ]:
ct = pd.crosstab(work["cluster"], work["label"])
chi2, p_value, _, _ = chi2_contingency(ct)
cramers_v = np.sqrt(chi2 / (ct.to_numpy().sum() * (min(ct.shape) - 1)))
purity = ct.max(axis=1).sum() / ct.to_numpy().sum()
association = pd.Series({
    "ARI (cụm vs nhãn)": adjusted_rand_score(work["label"], work["cluster"]),
    "Cramér's V": cramers_v,
    "p-value χ²": p_value,
    "Độ thuần (purity)": purity,
    "Mốc: tỷ lệ lớp đông nhất": work["label"].value_counts(normalize=True).max(),
})
association.to_frame("Giá trị").to_csv(RESULTS_DIR / "cluster_label_association.csv")
summary["% Fake"] = work.groupby("cluster")["label"].apply(lambda s: (s == 0).mean() * 100)
summary.to_csv(RESULTS_DIR / "cluster_summary.csv")
subject_table = pd.crosstab(work["cluster"], work["subject"])
subject_table.to_csv(RESULTS_DIR / "cluster_subjects.csv")
display(association.to_frame("Giá trị").round(5))
display(subject_table)
display(summary[["Chủ đề gợi ý", "Số bài", "% Fake"]].round(2))

## 7. Lưu mô hình và kiểm chứng tái sử dụng

`models/topic_clustering.joblib` chứa TF-IDF, SVD, Normalizer và K-Means đã fit. Đầu vào là `clean_text` từ notebook 3. Với bài thô, gọi `preprocess_article(title, text)` trước.

Dùng `predict_topics` để văn bản rỗng/toàn từ ngoài từ vựng nhận `-1`. K-Means vẫn ép mỗi bài có đặc trưng vào một cụm; bài ở ranh giới cần đọc thêm.

In [ ]:
pipeline = fitted_pipeline(topic_features, kmeans)
model_path = MODELS_DIR / "topic_clustering.joblib"
joblib.dump(pipeline, model_path)
loaded = joblib.load(model_path)
check_indices = np.linspace(0, len(work) - 1, 30, dtype=int)
with threadpool_limits(limits=4):
    predictions = predict_topics(loaded, work.iloc[check_indices]["clean_text"])
np.testing.assert_array_equal(predictions, kmeans.labels_[check_indices])
assert predict_topics(loaded, ["", "zzzzunknownword"] ).tolist() == [-1, -1]
assert len(assignments) == len(df)
assert int((assignments["cluster"] >= 0).sum()) == len(work)
assert summary["Số bài"].sum() == len(work)
metadata = {
    "tfidf_dtype": "float64", "selected_k": K, "seed": SEED, "seeds": list(SEEDS), "n_init": N_INIT,
    "k_values": list(K_VALUES), "lsa_dimensions": DIMENSIONS,
    "sample_size": len(sample_indices), "clustered_articles": len(work),
    "excluded_articles": len(excluded), "input": "notebook-3 clean_text",
    "selection": "manual choice after inspecting k_scores, k_runs and diagnostic plots",
    "labels_used_for_selection": False,
}
(RESULTS_DIR / "run_config.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã lưu:", model_path)
print("Kiểm chứng: mô hình nạp lại khớp 30 bài, đầu vào không có đặc trưng nhận -1, tổng số bài khớp.")

## Nhận xét tham khảo từ lần chạy cũ (K=10)

**Cần cập nhật phần này theo K bạn chọn thủ công và các bảng của lần chạy mới.** Các số liệu và tên cụm dưới đây thuộc lần chạy cũ.

**Lần chạy cũ chọn K=10** trên 38.820 bài; một bài không còn đặc trưng nhận `-1`. Với mẫu 3.000 bài, cosine silhouette trong TF-IDF là **0,01234**, ARI ổn định qua ba seed là **0,9889**, silhouette Euclidean trong LSA là **0,05791**.

Silhouette thấp cho thấy các chủ đề còn chồng lấn. K=12/14 có điểm cosine silhouette trung bình cao hơn nhưng ARI ổn định chỉ 0,719/0,706, thấp hơn K=10. Đây là một đánh đổi để cân nhắc khi chọn K thủ công.

Đọc từ khóa và bài đại diện, có thể gợi ý tên cụm như sau (mã cụm thuộc lần chạy cấu hình hiện tại, không phải nhãn ngữ nghĩa cố định):

| Cụm | Tên gợi ý khi đọc kết quả | Nhận xét |
|---|---|---|
| 0 | Trung Đông – Iran/Syria – quân sự | Chủ đề quốc tế |
| 1 | Chính trị – xã hội Mỹ, bình luận | Rộng, còn ảnh hưởng văn phong |
| 2 | Chính quyền – pháp luật – kinh tế | Rộng, còn trộn nhiều mảng |
| 3 | Hillary Clinton – email – Clinton Foundation | Nhóm nhân vật/sự kiện |
| 4 | Bầu cử sơ bộ Mỹ – ứng viên | Chủ đề bầu cử |
| 5 | Cảnh sát – súng đạn – tội phạm | Chủ đề xã hội |
| 6 | Thuế – y tế – Obamacare | Chính sách trong nước |
| 7 | Nga – FBI – điều tra | Chủ đề điều tra/chính trị |
| 8 | Triều Tiên – hạt nhân | Chủ đề quốc tế |
| 9 | Châu Âu – Brexit | Chủ đề quốc tế |

Các tên trên là diễn giải để kiểm tra, không phải kết quả chấm nhãn chủ đề. CSV vẫn giữ từ khóa gợi ý để có thể nhận diện cụm nếu chạy lại đổi mã.

**Các bước phân tích:** biểu diễn unigram + bigram, lọc từ văn phong, khảo sát K qua nhiều seed, chọn K thủ công, xem trọng số từ khóa và bài gần tâm/bài ranh giới, lưu mô hình để tái sử dụng.

**Giới hạn:** vẫn có cụm rộng và bài đa chủ đề; bộ lọc cũng có thể bỏ từ mang nội dung. Chưa có nhãn chủ đề chuẩn hoặc người chấm độc lập để đo độ chính xác chủ đề. ARI ổn định chỉ kiểm tra độ nhạy theo seed; liên hệ với Fake/True vẫn có thể do thiên lệch nguồn/chuyên mục. Đây là đánh giá nội tại trên dữ liệu khám phá, chưa kiểm tra khái quát hóa trên nguồn mới. Nếu đổi cấu hình và chạy lại, cần cập nhật phần nhận xét này theo các bảng mới.
